# 1. Import the necessary libraries

In [4]:
from mesh.core import *
from tuners import fine_tune_mesh, fine_tune_mesh_old, fine_tune_nsga2

from pymoo.indicators.hv import HV

import contextlib
import io
import optuna

# Adds the parent directory to sys.path
import sys
sys.path.append('../')
from problems.benchmark_problems import get_problem

# 2. Fine tuning configuration by hypervolume

In [ ]:
#################### CUSTOMIZABLE ####################

# Number of processes to execute the experiments in parallel
workers = 20

# Objective function configuration (function name, number of objectives, number of decision variables)
experiments = [
    ('zdt1', 2, 10, None), ('zdt2', 2, 10, None), ('zdt3', 2, 10, None), ('zdt4', 2, 10, None), ('zdt6', 2, 10, None),
    ('dtlz1', 3, 10, None), ('dtlz2', 3, 10, None), ('dtlz3', 3, 10, None), ('dtlz4', 3, 10, None), ('dtlz5', 3, 10, None), ('dtlz6', 3, 10, None), ('dtlz7', 3, 10, None),
    ('wfg1', 3, 10, 6), ('wfg2', 3, 10, 6), ('wfg3', 3, 10, 6), ('wfg4', 3, 10, 6), ('wfg5', 3, 10, 6), ('wfg6', 3, 10, 6), ('wfg7', 3, 10, 6), ('wfg8', 3, 10, 6),
    ('wfg9', 3, 10, 6)]

# Algorithm configuration
population_size = 100 # Population size
max_iteration = None # Number of iterations/generations
max_fitness_eval = 15000 # Maximum number of fitness evaluations
random_state = None # Set a seed for random numbers (not used if it is None)

tuning_folder = f'../hyperparams' # Folder to get the tuned parameters

# Fine tuning configuration
n_trials = 30 # Number of trials for Optuna
n_steps = 5 # Number of rounds in a trial
optuna_pruner = optuna.pruners.NopPruner() # Optuna's pruner to prune bad trials according to a metric
# optuna_pruner = optuna.pruners.WilcoxonPruner(p_threshold=0.05, n_startup_steps=n_steps//2)
hypercube_vertex = 100 # Consider the vertex of the hypercube
######################################################

ref_point = [np.array([hypercube_vertex] * exp[1]) for exp in experiments] # Reference point for hypervolume calculation
indicators = [HV(ref_point=rf) for rf in ref_point] # Define the indicator to calculate the volume

# 3 Define auxiliar functions

In [ ]:
def get_fitness_function(func_name, objective_dim, position_dim, wfg_k):
	func, position_min_value, position_max_value = get_problem(func_name, n_var=position_dim, n_obj=objective_dim, wfg_k=wfg_k)
	return func, objective_dim, position_dim, position_min_value, position_max_value

# Function to capture errors during parallel execution without stopping the other processes
def safe_run(run_function, params):
	try:
		f = io.StringIO()
		with contextlib.redirect_stdout(f):
			status = run_function(*params)
		return status, f.getvalue()
	except Exception as e:
		return f'Error: {str(e)}'

# 3. Apply fine tuning in the algorithms (in parallel)

## 3.1 Tuning MESH

In [ ]:
#################### CUSTOMIZABLE ####################

# MESH fixed parameters
mesh_memory_size = population_size # Maximum number of particles in memory
mesh_global_best_type = [0,1] # 0 -> Sigma method (G1) | 1 -> Sigma method in fronts (G2)
mesh_dm_pool_type = [0,1,2] # 0 -> Sampling from memory (S1) | 1 -> Sampling from population (S2) | 2 -> Sampling from memory and population (S3)
mesh_differential_evolution_type = [0,1,2,3,4] # 0 -> DE\rand\1\Bin (D1) | 1 -> DE\rand\2\Bin (D2) | 2 -> DE/Best/1/Bin (D3) | 3 -> DE/Current-to-best/1/Bin (D4) | 4 -> DE/Current-to-rand/1/Bin (D5)
######################################################

In [ ]:
# Set the list of parameters
params_list = [
  	[(F'MESH_G{gb_type+1}S{pool_type+1}D{de_type+1}_{exp[0]}_{exp[1]}_{exp[2]}', tuning_folder, max_fitness_eval, population_size, random_state),
     (n_trials, n_steps, optuna_pruner),
	 get_fitness_function(exp[0], exp[1], exp[2], exp[3]),
	 (mesh_memory_size, gb_type, pool_type, de_type),
	 indicators[i]
   ]
     for i, exp in enumerate(experiments)
     for gb_type in mesh_global_best_type
	 for pool_type in mesh_dm_pool_type
	 for de_type in mesh_differential_evolution_type
]

# Execute in parallel
Parallel(n_jobs=workers)(delayed(safe_run)(fine_tune_mesh, params) for params in params_list)

## Tuning MESH (previous implementation)

In [ ]:
# Set the list of parameters
params_list = [
  	[(F'OLD_MESH_G{gb_type+1}S{pool_type+1}D{de_type+1}_{exp[0]}_{exp[1]}_{exp[2]}', tuning_folder, max_fitness_eval, population_size, random_state),
     (n_trials, n_steps, optuna_pruner),
	 get_fitness_function(exp[0], exp[1], exp[2], exp[3]),
	 (mesh_memory_size, gb_type, pool_type, de_type),
	 indicators[i]
   ]
     for i, exp in enumerate(experiments)
     for gb_type in mesh_global_best_type
	 for pool_type in mesh_dm_pool_type
	 for de_type in mesh_differential_evolution_type
]

# Execute in parallel
Parallel(n_jobs=workers)(delayed(safe_run)(fine_tune_mesh_old, params) for params in params_list)

## 3.2 Tuning NSGA2

In [ ]:
# Set the list of parameters
params_list = [
  	[(F'NSGA2_{exp[0]}_{exp[1]}_{exp[2]}', tuning_folder, max_fitness_eval, population_size, random_state),
    (n_trials, n_steps, optuna_pruner),
	 get_fitness_function(exp[0], exp[1], exp[2], exp[3]),
	 (),
	 indicators[i]
   ]
     for i, exp in enumerate(experiments)
]

# Execute in parallel
Parallel(n_jobs=workers)(delayed(safe_run)(fine_tune_nsga2, params) for params in params_list)